In [1]:
# from 01-data.ipynb
import requests

URL_PREFIX='https://datatalks.club/faq'

path = '/json/courses.json'
response = requests.get(url=f'{URL_PREFIX}{path}')
courses_raw = response.json()

documents = []
for course in courses_raw:
    course_path = f'{URL_PREFIX}{course['path']}'
    course_response = requests.get(url=course_path)
    course_response.raise_for_status()
    course_faqs = course_response.json()

    documents.extend(course_faqs)

print(f'Total number of faq documents: {len(documents)}')

Total number of faq documents: 1350


In [2]:
# from 02-index-search.ipynb
from minsearch import Index

index = Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2.0, 'section':0.5}
    filter_dict={'course':course}

    return index.search(
        query=question,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=5
    )

In [3]:
# from 03-build-prompt.ipynb
SYSTEM_PROMPT = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append(f'Q: {doc['question']}')
        lines.append(f'A: {doc['answer']}')
        lines.append('')
    
    return '\n'.join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results=search_results)
    prompt =  USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

    return prompt.strip()

question = 'I just discovered the course. Can I join now?'
search_results = search(question)
prompt = build_prompt(question=question, search_results=search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [4]:
# input the prompt into an llm and return the response
from openai import OpenAI
from dotenv import load_dotenv
import os
from pprint import pprint

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

openai_client = OpenAI(api_key=api_key)

def llm(openai_client, system_prompt, user_prompt, model='gpt-4o-mini'):
    # separate the fixed instruction (system prompt) and the prompt (user prompt) which changes with every request
    payload = [
        {'role':'developer', 'content':system_prompt},
        {'role':'user', 'content':user_prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=payload
    )

    return response

response = llm(openai_client=openai_client, system_prompt=SYSTEM_PROMPT, user_prompt=prompt)
pprint(response.output_text) # response.output_text is a shortcut for response.output[0].content[0].text

# calculate the cost of the query - refer to https://developers.openai.com/api/docs/models/ for input and output token cost per million of tokens
input_token_cost = 0.15/1_000_000
output_token_cost = 0.60/1_000_000
cost = response.usage.input_tokens * input_token_cost + response.usage.output_tokens * output_token_cost
print(cost)

('Yes, you can still join the course. However, if you want to receive a '
 'certificate, you need to submit your project while submissions are still '
 'being accepted.')
9.974999999999999e-05


In [6]:
# wire the search, build_prompt and llm functions together to make a full RAG pipeline
def rag(query, model='gpt-4o-mini'):
    search_results = search(question=query)
    user_prompt = build_prompt(question=query, search_results=search_results)
    response = llm(openai_client=openai_client, system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, model=model)
    return response

response = rag(query="How do I get a certificate?", model='gpt-5.4-mini')
print(response.output_text)

To get a certificate, you need to finish the course with a **live cohort** and **pass the Capstone project**.

A few important points:
- **Self-paced mode does not award certificates**
- **Homework is not mandatory** for the certificate
- You must also **peer-review 3 capstones** during the live course period

If you want your real name on the certificate, update your **official name** in the course profile.
